# AutoGen

**Domain:** Agentic AI  ·  **runnable:** yes

A refresher on **AutoGen** — Microsoft's framework for building applications out of **multiple agents that solve a task by talking to each other**. You define a few agents (an LLM-backed assistant, a user proxy that can run code, maybe a critic), drop them into a *conversation*, and AutoGen drives the back-and-forth: each agent receives the running message log, generates a reply, and passes the turn on until a termination condition fires.

Where [[crewai]] frames work as *roles + tasks on an assembly line* and [[langgraph]] hands you a low-level *state graph*, AutoGen's organizing idea is **conversation**: orchestration is just deciding who speaks next and when to stop.

## 1. What & Why

**AutoGen** (Microsoft Research) is a framework for **conversational multi-agent** systems. The unit you compose is an *agent that can send and receive chat messages*; an application is a **conversation** between two or more of them. The framework owns the loop — relaying messages, invoking each agent's `generate_reply`, executing any code an agent emits, and checking for termination — so you focus on *which* agents exist and *how* the conversation is structured.

**The problem it solves.** Many real tasks are iterative: write code → run it → read the error → fix it → re-run. A single LLM call can't close that loop, and hand-rolling the "model writes code, something executes it, feed the result back" cycle is fiddly and easy to get wrong. AutoGen's killer pattern is exactly this: an **AssistantAgent** that writes code paired with a **UserProxyAgent** that *executes* it and reports back, looping automatically until the task is done or a stop condition is hit. The same machinery generalizes to debate, review, and tool-use patterns.

**When to reach for it.** Tasks that are naturally a *dialogue* — code-gen-and-execute, solver-plus-critic, brainstorm-then-converge, or a small "team" where the next speaker depends on what was just said. **When not to:** a fixed linear pipeline (use [[langchain]] LCEL), a strict role-based workflow (use [[crewai]]), or a graph where you need explicit control over state, branching, and persistence (use [[langgraph]]). Free-form conversation is flexible but harder to constrain and cost-bound.

## 2. Mental Model

Think of AutoGen as **a group chat where the framework is the moderator.**

- Every **agent** is a chat participant with a `name` and a `generate_reply(messages)` method. Given the conversation so far, it returns its next message (an LLM call, a canned reply, the result of executing code, or a tool call).
- A **two-agent chat** is a ping-pong: `A.initiate_chat(B, message=...)` sends the opening line, then A↔B alternate — each `receive`s the other's message, `generate_reply`s, and `send`s back — until a **termination** rule trips.
- A **group chat** adds a **manager**: after each message the `GroupChatManager` picks the next speaker (round-robin, an LLM "who should go next?" selector, or your own function), appends their reply to the shared transcript, and repeats.
- The classic duo is **AssistantAgent ✍️ writes** a code block and **UserProxyAgent ▶️ runs** it, feeding stdout/errors back as the next message — an automatic write→execute→fix loop.

```
initiate_chat ─▶ [ Assistant ] ⇄ [ UserProxy(executes code) ]
                      ▲                      │
                      └──── stdout / error ──┘   (loop until termination)

GroupChat:   Manager picks next speaker ─▶ appends reply ─▶ repeat ─▶ stop
```

The whole framework is essentially: *maintain a message log, decide who speaks next, call their `generate_reply`, check if we should stop.*

## 3. Key Concepts

| Concept | What it is |
|---|---|
| **`ConversableAgent`** | The base agent. Holds a `name`, an optional `system_message`, an LLM config (`llm_config`), and the `send` / `receive` / `generate_reply` methods. Everything else subclasses it. |
| **`AssistantAgent`** | A `ConversableAgent` preset to be the **LLM brain**: by default it plans and writes code/answers in reply to the conversation. No code execution itself. |
| **`UserProxyAgent`** | Stands in for the human and acts as the **executor**: it can run code blocks the assistant produced, relay results, and optionally ask a real human (`human_input_mode`). |
| **`initiate_chat`** | Kicks off a two-agent conversation: `a.initiate_chat(b, message=...)`. Returns a `ChatResult` with the full message history and any summary. |
| **`GroupChat` / `GroupChatManager`** | Multi-agent orchestration: `GroupChat` holds the agent list + shared messages; the `Manager` selects the next speaker each turn (`round_robin`, `auto` LLM-selection, or custom). |
| **`human_input_mode`** | `"NEVER"` (fully autonomous), `"TERMINATE"` (ask the human only when the agent would otherwise stop), or `"ALWAYS"` (prompt every turn). |
| **Termination** | How a chat ends: `is_termination_msg` (e.g. message contains `"TERMINATE"`), `max_consecutive_auto_reply`, or in v0.4 explicit `TextMentionTermination` / `MaxMessageTermination` conditions. |
| **Code execution** | The user proxy runs code via a configured executor (local command line or Docker). Off-by-default safety matters — see Gotchas. |

> **Two major versions.** The classic API (`pyautogen`, ≤ v0.2) uses `ConversableAgent` / `GroupChat` synchronously. The rewrite (**v0.4+**, packages `autogen-agentchat` + `autogen-core` + `autogen-ext`) is **async**, with model *clients*, **Teams** (`RoundRobinGroupChat`, `SelectorGroupChat`), and composable termination conditions. The *mental model is identical*; the imports and method names differ. Pin your version and read the matching docs.

## 4. Setup

AutoGen now ships as a few focused packages (v0.4+). The agent-team layer most people want is `autogen-agentchat`, plus an extension package for model clients:

```bash
# Current generation (v0.4+), async API
pip install -U "autogen-agentchat" "autogen-ext[openai]"

# Classic generation (v0.2), synchronous ConversableAgent / GroupChat
pip install "pyautogen"            # imported as `import autogen`

export OPENAI_API_KEY=...          # AutoGen has no offline default model
```

Requires **Python 3.10+**. There is no built-in free model — every real run needs a provider key (OpenAI, Azure OpenAI, or any OpenAI-compatible / local endpoint via a custom `base_url`). For safe code execution, also install Docker (recommended) or opt into a local executor.

Examples 1 & 2 below are **pure standard-library** reimplementations of AutoGen's conversation core, so they run **offline with no key**. Example 3 shows the real library and is gated behind an install + key check.

In [ ]:
# Setup check — examples 1 & 2 need nothing; example 3 needs the library + a key.
import importlib.util, os

has_autogen_v04 = importlib.util.find_spec("autogen_agentchat") is not None
has_autogen_v02 = importlib.util.find_spec("autogen") is not None
has_key         = bool(os.getenv("OPENAI_API_KEY"))

print("autogen-agentchat (v0.4+) installed:", has_autogen_v04)
print("pyautogen (v0.2) installed:         ", has_autogen_v02)
print("OPENAI_API_KEY present:             ", has_key)
print("\nExamples 1 & 2 run regardless (stdlib only).")

## 5. Worked Examples

### Example 1 — The two-agent chat loop, from scratch

AutoGen's core is small: an agent has a `name` and a `generate_reply(messages)`; `initiate_chat` just alternates `send`/`receive` between two agents until a termination rule fires. Below we reimplement that loop with a deterministic *fake* LLM (no key, fully offline) so you can see exactly what `assistant ↔ user_proxy` does — including the **write-code → execute → feed-result-back** pattern that makes AutoGen click.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable

# --- the conversation primitive: an agent is name + generate_reply ---
@dataclass
class ConversableAgent:
    name: str
    reply_fn: Callable[[list], str]          # given the transcript, return next message

    def generate_reply(self, messages: list) -> str:
        return self.reply_fn(messages)


def initiate_chat(a: ConversableAgent, b: ConversableAgent,
                  message: str, is_termination, max_turns: int = 8) -> list:
    """Two-agent ping-pong, exactly what AutoGen's initiate_chat does."""
    transcript = [{"name": a.name, "content": message}]
    speaker, other = b, a                    # b replies to a's opening line first
    for _ in range(max_turns):
        reply = speaker.generate_reply(transcript)
        transcript.append({"name": speaker.name, "content": reply})
        if is_termination(reply):
            break
        speaker, other = other, speaker      # swap turns
    return transcript


# --- a deterministic stand-in for the AssistantAgent's LLM ---
def assistant_brain(messages):
    last = messages[-1]["content"]
    if "Task:" in last:                      # first turn: propose code
        return "Here is code to run:\n```python\nprint(6 * 7)\n```"
    if "OUTPUT: 42" in last:                 # saw the execution result -> done
        return "The script printed 42. TERMINATE"
    return "Let me reconsider..."

# --- the UserProxyAgent: it EXECUTES code blocks and reports the result ---
def user_proxy_executor(messages):
    last = messages[-1]["content"]
    if "```python" in last:
        snippet = last.split("```python")[1].split("```")[0].strip()
        env = {}
        import io, contextlib
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            exec(snippet, env)               # runs the assistant's code
        return f"OUTPUT: {buf.getvalue().strip()}"
    return "(no code to run)"

assistant  = ConversableAgent("assistant",  assistant_brain)
user_proxy = ConversableAgent("user_proxy", user_proxy_executor)

chat = initiate_chat(
    user_proxy, assistant,
    message="Task: compute 6 * 7 by writing and running Python.",
    is_termination=lambda m: "TERMINATE" in m,
)

for turn in chat:
    print(f"[{turn['name']:>10}] {turn['content']}")

### Example 2 — A `GroupChat` with a speaker-selecting manager

Beyond two agents, AutoGen uses a **`GroupChatManager`** that, after every message, picks the next speaker and appends their reply to one shared transcript. Here we build a tiny group chat with three agents (planner → coder → critic) and a round-robin manager, plus a termination check — exactly the shape of `GroupChat` + `GroupChatManager` (or v0.4's `RoundRobinGroupChat` team).

In [ ]:
def run_group_chat(agents, manager_select, opening, is_termination, max_rounds=9):
    """A GroupChatManager: shared transcript, pick next speaker, repeat until stop."""
    transcript = [{"name": "user", "content": opening}]
    for round_i in range(max_rounds):
        speaker = manager_select(agents, round_i)       # who talks next
        reply = speaker.generate_reply(transcript)
        transcript.append({"name": speaker.name, "content": reply})
        if is_termination(reply):
            break
    return transcript

def round_robin(agents, round_i):
    return agents[round_i % len(agents)]

# three role-flavored fake agents sharing one conversation
def make_brain(role):
    def brain(messages):
        return {
            "planner": "PLAN: 1) parse input  2) sum the numbers  3) report.",
            "coder":   "def total(xs): return sum(xs)   # implements the plan",
            "critic":  "Looks correct and handles empty lists. APPROVED — TERMINATE",
        }[role]
    return brain

agents = [
    ConversableAgent("planner", make_brain("planner")),
    ConversableAgent("coder",   make_brain("coder")),
    ConversableAgent("critic",  make_brain("critic")),
]

chat = run_group_chat(
    agents, round_robin,
    opening="Build and review a function that sums a list of numbers.",
    is_termination=lambda m: "TERMINATE" in m,
)

for turn in chat:
    print(f"[{turn['name']:>8}] {turn['content']}")
print(f"\nConversation ran {len(chat)-1} agent turn(s) before terminating.")

### Example 3 — The real thing: `AssistantAgent` + `UserProxyAgent`

This is the production pattern with real AutoGen. It calls a live model, so it runs **only** if the library is installed *and* `OPENAI_API_KEY` is set; otherwise it prints the exact code you'd write. Both the classic (v0.2) and current (v0.4+) shapes are shown — note they're the same idea as Examples 1–2: an assistant that writes code and a user proxy that executes it, looping until `TERMINATE`.

In [ ]:
SNIPPET_V02 = '''
# --- Classic AutoGen (pyautogen, v0.2): synchronous ---
import autogen

llm_config = {"model": "gpt-4o-mini"}   # reads OPENAI_API_KEY from the env

assistant = autogen.AssistantAgent(name="assistant", llm_config=llm_config)
user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    human_input_mode="NEVER",                 # fully autonomous
    max_consecutive_auto_reply=5,
    is_termination_msg=lambda m: "TERMINATE" in (m.get("content") or ""),
    code_execution_config={"work_dir": "coding", "use_docker": False},
)

# assistant writes code -> user_proxy runs it -> result feeds back, until TERMINATE
user_proxy.initiate_chat(assistant, message="Plot is overkill; just compute 6*7 in Python.")
'''

SNIPPET_V04 = '''
# --- Current AutoGen (autogen-agentchat, v0.4+): async, teams ---
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

async def main():
    client = OpenAIChatCompletionClient(model="gpt-4o-mini")
    planner = AssistantAgent("planner", model_client=client,
                             system_message="Plan, then say TERMINATE when done.")
    coder   = AssistantAgent("coder", model_client=client)
    team = RoundRobinGroupChat([planner, coder],
                               termination_condition=TextMentionTermination("TERMINATE"))
    result = await team.run(task="Write and explain a function that sums a list.")
    print(result.messages[-1].content)

asyncio.run(main())
'''

import importlib.util, os
have_lib = (importlib.util.find_spec("autogen") is not None
            or importlib.util.find_spec("autogen_agentchat") is not None)

if have_lib and os.getenv("OPENAI_API_KEY"):
    print("Library + key present. In a real session you would run one of the snippets below.")
    print("(Skipping the live model call here to keep the notebook fast and deterministic.)")
else:
    print("autogen not installed or OPENAI_API_KEY unset — showing the code shape only.\n")
    print("# ===== v0.2 (classic) =====")
    print(SNIPPET_V02)
    print("# ===== v0.4+ (current) =====")
    print(SNIPPET_V04)

## 6. Gotchas & Pitfalls

- **v0.2 vs v0.4 are different libraries.** `pip install pyautogen` gives the classic `import autogen` API; `pip install autogen-agentchat` gives the async v0.4 API. Tutorials, imports, and method names don't transfer between them. Check which version a snippet targets before copy-pasting, and pin it in your requirements. (Confusingly, an unrelated PyPI package also grabbed the name `autogen` for a while — install the exact package you mean.)
- **Code execution is arbitrary code execution.** A `UserProxyAgent` with code execution runs whatever the LLM writes. Run it in **Docker** (`use_docker=True`) or a sandbox, never blindly on your machine, and review what you let it touch — especially with untrusted prompts.
- **Conversations don't stop themselves.** Without a real termination rule a chat ping-pongs until it hits `max_consecutive_auto_reply` / `max_turns` — or burns tokens forever. Always set `is_termination_msg` (or a v0.4 termination condition) *and* a turn/message cap.
- **`human_input_mode` defaults bite.** The classic `UserProxyAgent` defaults to `"ALWAYS"`, which blocks waiting for console input — great for notebooks, a hang in a server. Set `"NEVER"` for autonomous runs.
- **The "TERMINATE" handshake is brittle.** Termination often keys on the model literally emitting `TERMINATE`. If your system prompt doesn't instruct it to, the chat won't end; if the word appears mid-explanation, it ends early. Make the stop token explicit in the prompt.
- **Free-form ≠ free.** Every turn is an LLM call over the *growing* transcript, so cost scales with conversation length. Group chats with auto speaker-selection add an extra selection call per turn. Watch token usage and cap rounds.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-offs vs AutoGen |
|---|---|---|
| **AutoGen** | **Conversational** multi-agent tasks: code-write-and-execute loops, solver+critic, debate, dynamic "who speaks next" teams | Free-form chat is flexible but harder to constrain/cost-bound; two API generations to keep straight; orchestration is implicit in the conversation |
| **[[crewai]]** | **Role + task** pipelines you can describe in plain language; clear sequential / hierarchical structure | More opinionated and structured; less natural for open-ended back-and-forth or emergent dialogue |
| **[[langgraph]]** | **Explicit** stateful graphs: custom routing, branching, cycles, persistence, human-in-the-loop, resume | Lower-level — you wire nodes/edges/state yourself; AutoGen hides that behind "just have them talk" |
| **[[langchain]] (LCEL)** | **Linear** dataflow: `retrieve → prompt → model → parse`; no multi-agent dialogue needed | A pipeline, not a conversation; reach for it when one agent + tools is enough |
| **[[openai-agents-sdk]]** | Lightweight agents + handoffs tightly bound to OpenAI models/tooling | Vendor-leaning; AutoGen is model-agnostic and group-chat-centric |

**Rule of thumb:** if the solution *is a dialogue* (especially "write code, run it, fix it"), AutoGen is the natural fit. If it's a fixed workflow or you need tight control over state and branching, prefer CrewAI or LangGraph.

## 8. Resources

- **Official site & docs (v0.4+)** — https://microsoft.github.io/autogen/stable/
- **AgentChat tutorial (current API)** — https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/index.html
- **Migration guide (v0.2 → v0.4)** — https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/migration-guide.html
- **Classic v0.2 docs** — https://microsoft.github.io/autogen/0.2/
- **GitHub repository** — https://github.com/microsoft/autogen
- **AutoGen Studio (low-code UI)** — https://microsoft.github.io/autogen/stable/user-guide/autogenstudio-user-guide/index.html
- **Paper: "AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation"** — https://arxiv.org/abs/2308.08155

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
class Termination:
    ...


def run_chat(agents, opening, select, terminate, max_turns=10):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE